# Feature Engineering - определение факта сделки на Авито
**Задача:** бинарная классификация `is_deal`.

Цель - подготовить признаки на уровне объявления и сравнить их вклад в качество логистической регрессии.

## 1. Загрузка данных и базовые проверки
### 1.1. Библиотеки и вспомогательные функции

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
for path in sorted(Path("data").rglob("*.csv")):
    print(path)


def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

### 1.2. Чтение исходных таблиц
Пути соответствуют структуре репозитория команды. Проверяем размеры, типы, пропуски и дубликаты.

In [ ]:
DATA = Path("data/data_wthout_anomailes")

listings = pd.read_csv(DATA / "listings_wthout_anomailes.csv")
events = pd.read_csv(DATA / "events_wthout_anomailes.csv")
messages = pd.read_csv(DATA / "messages_wthout_anomailes.csv")
surveys = pd.read_csv(DATA / "surveys_wthout_anomailes.csv")

verified = pd.read_csv("data/verified.csv")

tables = {
    "listings": listings,
    "events": events,
    "messages": messages,
    "surveys": surveys,
    "verified": verified
}

for name, df in tables.items():
    print(f"\n{name}: {df.shape}")
    print(df.dtypes)
    print("Дубликатов:", df.duplicated().sum())
    print("Пропуски:")
    print(df.isna().sum())

In [ ]:
# Проверка ключей перед построением признаков.
assert listings["listing_id"].is_unique, "В listings есть повторяющиеся ID"
assert verified["listing_id"].is_unique, "В verified есть повторяющиеся ID"
assert verified["is_deal"].isin([0, 1]).all(), "Некорректная разметка"

## 2. Listings - признаки объявления
### 2.1. Исследование данных
Отбираем размеченные объявления, присутствующие в очищенной таблице. Идентификаторы не используются как признаки модели.

In [ ]:
display(listings.head())
display(listings.describe().T)

for col in ["seller_type", "category", "region", "close_type"]:
    print(f"\n{col}:")
    print(listings[col].value_counts(dropna=False))

verified_clean = verified[
    verified["listing_id"].isin(listings["listing_id"])
]

print("Всего verified:", len(verified))
print("После очистки:", len(verified_clean))
print(verified_clean["is_deal"].value_counts())

### 2.2. Нормализация региона
Приводим разные написания одного города к единому значению. Рабочая таблица `features` начинается с копии listings.

In [ ]:
features = listings.copy()

features["region"] = (
    features["region"]
    .str.strip()
    .str.lower()
    .str.replace(r"^г\.\s*", "", regex=True)
)

region_map = {
    "moscow": "москва",
    "масква" : "москва",
    "мск" : "москва",
    "спб": "санкт-петербург",
    "санкт-питербург" : "санкт-петербург",
    "msk" : "москва",
    "питер": "санкт-петербург",
    "санкт петербург": "санкт-петербург",
    "екб": "екатеринбург",
    "новосиб" : "новосибирск",
    "н. новгород": "нижний новгород"
}

features["region"] = features["region"].replace(region_map)

print(features["region"].value_counts())

### 2.3. Восстановление категорий и индикатор пропусков
Проверяем однозначность соответствия заголовка категории. Пропуски длины описания оставляем для импутации в модели и отмечаем бинарным флагом.

In [ ]:
print("Всего объявлений:", len(features))
print("Уникальных заголовков:", features["title"].nunique())

display(features["title"].value_counts().head(20))

display(
    features.loc[
        features["category"].isna(),
        ["title", "price", "category"]
    ].head(30)
)

title_categories = (
    features.dropna(subset=["category"])
    .groupby("title")["category"]
    .nunique()
)

print(title_categories.value_counts())

features["description_missing"] = (
    features["description_length"].isna().astype(int)
)

In [ ]:
# Восстановление категории по известным заголовкам.

title_to_category = (
    features.dropna(subset=["category"])
    .groupby("title")["category"]
    .first()
    .to_dict()
)

features["category"] = features["category"].fillna(
    features["title"].map(title_to_category)
)

print("Осталось пропусков:", features["category"].isna().sum())

### 2.4. Заголовки и относительная цена
Выделяем состояние товара и другие суффиксы, затем убираем их для получения базового названия товара. Медианная цена группы позволяет сравнивать разные товары на общей шкале.

In [ ]:
title = features["title"].str.lower().str.strip()

features["is_used"] = title.str.endswith(" б/у").astype(int)
features["is_new"] = (
    title.str.endswith(" новый")
    & ~title.str.endswith(" как новый")
).astype(int)

features["is_like_new"] = title.str.endswith(" как новый").astype(int)
features["has_documents"] = title.str.endswith(" с документами").astype(int)
features["is_negotiable"] = title.str.endswith(" торг").astype(int)
features["is_urgent"] = title.str.endswith(" срочно").astype(int)
features["is_excellent"] = title.str.endswith(" отличное состояние").astype(int)
features["has_package"] = title.str.contains(
    r"\b(?:коробка|коробкой|упаковка|упаковкой)\b",
    case=False,
    na=False
).astype(int)

suffix_pattern = (
    r"\s+(?:в хорошем состоянии|отличное состояние|"
    r"с документами|как новый|б/у|новый|торг|срочно)$"
)

features["base_title"] = title.str.replace(
    suffix_pattern, "", regex=True
)

features["is_good_condition"] = (
    title.str.endswith(" в хорошем состоянии").astype(int)
)

print("Товарных групп:", features["base_title"].nunique())
print(features["base_title"].value_counts().describe())

print("До:", features["title"].nunique())
print("После:", features["base_title"].nunique())

display(features["base_title"].value_counts().head(20))

# относительная цена

features["product_median_price"] = (
    features.groupby("base_title")["price"].transform("median")
)

features["relative_price"] = (
    features["price"] / features["product_median_price"]
)

features["relative_current_price"] = (
    features["current_price"] / features["product_median_price"]
)

features["log_relative_price"] = np.log(
    features["relative_price"]
)

display(
    features[
        [
            "title",
            "base_title",
            "category",
            "price",
            "product_median_price",
            "relative_price"
        ]
    ].head(15)
)

print(features["relative_price"].describe())

print(
    features["base_title"].value_counts().describe()
)

### 2.5. Исследование ценовых признаков
Сопоставляем относительную цену с проверенной разметкой; группы цен используются только для анализа, не как признаки модели.

In [ ]:
unique_titles = sorted(
    title.dropna().unique(),
    key=len,
    reverse=True
)

suffixes = []

for full_title in unique_titles:
    for short_title in unique_titles:
        if full_title.startswith(short_title + " "):
            suffix = full_title[len(short_title):].strip()
            suffixes.append(suffix)
            break

display(pd.Series(suffixes).value_counts())

research = features.merge(
    verified,
    on="listing_id",
    how="inner",
    validate="one_to_one"
)

print(research.shape)

display(
    research.groupby("is_deal")[
        [
            "price",
            "relative_price",
            "relative_current_price",
            "log_relative_price"
        ]
    ].agg(["mean", "median", "std"]).T
)

research["price_group"] = pd.cut(
    research["relative_price"],
    bins=[0, 0.5, 0.8, 1.2, 1.5, np.inf],
    labels=[
        "Очень дешёвый",
        "Дешёвый",
        "Рыночная цена",
        "Дорогой",
        "Очень дорогой"
    ]
)

price_stats = (
    research.groupby("price_group", observed=True)
    .agg(
        total=("is_deal", "count"),
        deals=("is_deal", "sum"),
        deal_rate=("is_deal", "mean")
    )
)

display(price_stats)

price_stats["deal_rate"].plot(
    kind="bar",
    figsize=(10, 4),
    rot=20,
    title="Зависимость вероятности сделки от относительной цены"
)

plt.axhline(
    research["is_deal"].mean(),
    color="red",
    linestyle="--",
    label="Общая доля сделок"
)

plt.ylabel("Доля сделок")
plt.ylim(0, 1)
plt.legend()
plt.show()

### 2.6. Скидка, время жизни и способ закрытия
Считаем изменение цены и длительность размещения. Проверяем, связаны ли срок жизни и способ закрытия с фактом сделки; закрытие само по себе не гарантирует продажу.

In [ ]:
# Относительная величина снижения цены
features["discount_pct"] = (
    (features["price"] - features["current_price"])
    / features["price"]
)

# Снижалась ли цена к моменту закрытия
features["has_discount"] = (
    (features["current_price"] < features["price"])
    .astype(int)
    .where(features[["price", "current_price"]].notna().all(axis=1))
)

# Цена существенно выше медианы
features["is_overpriced"] = (
    (features["relative_price"] > 1.5)
    .astype(int)
    .where(features["relative_price"].notna())
)

# Логарифм абсолютной цены
features["log_price"] = np.log1p(features["price"])

features["published_at"] = pd.to_datetime(
    features["published_at"]
)

features["closed_at"] = pd.to_datetime(
    features["closed_at"]
)

features["lifetime_hours"] = (
    features["closed_at"] - features["published_at"]
).dt.total_seconds() / 3600

features["lifetime_days"] = features["lifetime_hours"] / 24

display(features["lifetime_days"].describe())

features["is_closed_first_day"] = (
    (features["lifetime_days"] <= 1)
    .astype(int)
    .where(features["lifetime_days"].notna())
)

features["is_closed_first_week"] = (
    (features["lifetime_days"] <= 7)
    .astype(int)
    .where(features["lifetime_days"].notna())
)

features["log_lifetime"] = np.log1p(
    features["lifetime_days"]
)

research = features.merge(
    verified, on="listing_id", how="inner"
)

display(
    research.groupby("is_deal")["lifetime_days"]
    .agg(["mean", "median", "std", "min", "max"])
)

research["lifetime_group"] = pd.cut(
    research["lifetime_days"],
    bins=[0, 1, 3, 7, 14, 29, np.inf],
    labels=[
        "До 1 дня",
        "1–3 дня",
        "3–7 дней",
        "7–14 дней",
        "14–29 дней",
        "Более 29 дней"
    ],
    include_lowest=True
)

lifetime_stats = (
    research.groupby("lifetime_group", observed=True)
    .agg(
        total=("is_deal", "count"),
        deals=("is_deal", "sum"),
        deal_rate=("is_deal", "mean")
    )
)

display(lifetime_stats)

display(
    research.groupby("close_type")
    .agg(
        total=("is_deal", "count"),
        deals=("is_deal", "sum"),
        deal_rate=("is_deal", "mean")
    )
)

In [ ]:
# Проверка связи длительности объявления со способом его закрытия.

display(pd.crosstab(
    research["lifetime_group"],
    research["close_type"]
))

In [ ]:
# Анализ только объявлений, снятых продавцом.

display(
    research[
        research["close_type"] == "снято продавцом"
    ]
    .groupby("lifetime_group", observed=True)["is_deal"]
    .agg(["count", "sum", "mean"])
)

### 2.7. Поведенческие признаки
Нормируем количество действий по просмотрам, чатам и времени жизни. `safe_divide` оставляет NaN, если знаменатель равен нулю; при расчёте дневной активности минимальный срок - один день.

In [ ]:
# Конверсии просмотров в контактные действия и глубина переписки.

features["messages_per_chat"] = safe_divide(
    features["messages_total"],
    features["chats_total"]
)

features["chats_per_view"] = safe_divide(
    features["chats_total"],
    features["views_total"]
)

features["phones_per_view"] = safe_divide(
    features["phone_views_total"],
    features["views_total"]
)

features["favorites_per_view"] = safe_divide(
    features["favorites_total"],
    features["views_total"]
)

features["contacts_total"] = (
    features["chats_total"]
    + features["phone_views_total"]
)

features["contacts_per_view"] = safe_divide(
    features["contacts_total"],
    features["views_total"]
)

In [ ]:
# Интенсивность активности за день.

days = features["lifetime_days"].clip(lower=1)

features["views_per_day"] = (
    features["views_total"] / days
)

features["chats_per_day"] = (
    features["chats_total"] / days
)

features["messages_per_day"] = (
    features["messages_total"] / days
)

features["contacts_per_day"] = (
    features["contacts_total"] / days
)

### 2.8. Относительные характеристики внутри товарной группы
Сравниваем активность, длину описания и время жизни с медианами похожих товаров. Не все экспериментальные признаки попадут в итоговый набор.

In [ ]:
# относительная популярность товара по просмотрам

features["views_per_day_median"] = (
    features.groupby("base_title")["views_per_day"]
    .transform("median")
)

features["relative_views_per_day"] = safe_divide(
    features["views_per_day"],
    features["views_per_day_median"]
)

# относительная популярность товара по сообщениям

features["median_messages_per_chat"] = (
    features.groupby("base_title")["messages_per_chat"]
    .transform("median")
)

features["relative_messages_per_chat"] = safe_divide(
    features["messages_per_chat"],
    features["median_messages_per_chat"]
)

# относительная популярность товара по длине описания

features["median_description_length"] = (
    features.groupby("base_title")["description_length"]
    .transform("median")
)

features["relative_description_length"] = safe_divide(
    features["description_length"],
    features["median_description_length"]
)

# относительная популярность товара lifetime товара

features["median_lifetime_days"] = (
    features.groupby("base_title")["lifetime_days"]
    .transform("median")
)

features["relative_lifetime_days"] = safe_divide(
    features["lifetime_days"],
    features["median_lifetime_days"]
)

### 2.9. Отсутствие просмотров и контактов
Отдельные бинарные признаки позволяют различать отсутствие событий и пропуски в относительных величинах.

In [ ]:
features["no_contacts"] = (
    features["contacts_total"] == 0
).astype(int)

features["no_chats"] = (
    features["chats_total"] == 0
).astype(int)

features["no_phone_views"] = (
    features["phone_views_total"] == 0
).astype(int)

features["no_views"] = (
    features["views_total"] == 0
).astype(int)

## 3. Surveys - ответы продавцов
### 3.1. Проверка ответов и исследование достоверности
Опрос доступен частным продавцам, снявшим объявление. Ответ - сигнал, но не целевая переменная: по сохранённым результатам 41 из 51 ответа «Продал на Авито» подтверждён разметкой. Намеренный обман из этих данных установить нельзя.

In [ ]:
display(surveys.head(10))

print("Размер:", surveys.shape)

print("\nПропуски:")
print(surveys.isna().sum())

print("\nВарианты ответов:")
print(surveys["answer"].value_counts(dropna=False))

print("\nДубликаты строк:")
print(surveys.duplicated().sum())

print("\nПовторяющиеся listing_id:")
print(surveys["listing_id"].duplicated().sum())

survey_conflicts = (
    surveys.groupby("listing_id")["answer"]
    .nunique()
)

display(survey_conflicts.value_counts())

# Разделение ответов по столбцам
answers_count = surveys["answer"].value_counts()

for answer_type in answers_count.index.tolist():
    surveys[f"answer_type_{answer_type}"] = (
        surveys["answer"] == answer_type
    ).astype(int)

# Мог ли продавец получить опрос
features["survey_eligible"] = (
    (features["seller_type"] == "private")
    & (features["close_type"] == "снято продавцом")
).astype(int)

# Есть ли непустой ответ на опрос
features["survey_answered"] = (
    features["listing_id"].isin(
        surveys.loc[surveys["answer"].notna(), "listing_id"]
    )
).astype(int)

# Сопоставление ответов с проверенной разметкой.
survey_research = (
    surveys[["listing_id", "answer"]]
    .drop_duplicates()
    .merge(verified_clean, on="listing_id", how="inner")
)

display(
    survey_research.groupby("answer")
    .agg(
        total=("is_deal", "count"),
        deals=("is_deal", "sum"),
        deal_rate=("is_deal", "mean")
    )
)

### 3.2. Объединение с объявлениями
Используем left join: объявления без опроса сохраняются, их индикаторы ответов равны нулю. `survey_eligible` отличает потенциально доступный опрос от недоступного.

In [ ]:
survey_features = [
    col for col in surveys.columns
    if col.startswith("answer_type_")
]

features = features.drop(
    columns=survey_features,
    errors="ignore"
).merge(
    surveys[["listing_id"] + survey_features],
    on="listing_id",
    how="left",
    validate="one_to_one"
)

features[survey_features] = (
    features[survey_features].fillna(0).astype(int)
)

print(features.shape)
print(features["listing_id"].is_unique)

display(
    features[
        survey_features
        + ["survey_eligible", "survey_answered"]
    ].sum()
)

## 4. Events - события покупателей
### 4.1. Исследование и проверка времени
Пропуски в user_id для изменения цены и в new_price для остальных действий закономерны. Время событий проверяется в data_clearing.ipynb. Если даты объявления неизвестны, для агрегации остаются события с корректным временем.

In [ ]:
display(events.head(15))

print("Размер:", events.shape)

print("\nТипы данных:")
print(events.dtypes)

print("\nПропуски:")
print(events.isna().sum())

print("\nДубликаты:")
print(events.duplicated().sum())

print("\nТипы событий:")
print(events["event_type"].value_counts(dropna=False))

print("\nУникальных покупателей:")
print(events["user_id"].nunique())

print("\nУникальных объявлений:")
print(events["listing_id"].nunique())

display(
    events.groupby("event_type")
    .agg(
        total=("listing_id", "count"),
        unique_listings=("listing_id", "nunique"),
        unique_users=("user_id", "nunique"),
        missing_users=("user_id", lambda x: x.isna().sum()),
        missing_prices=("new_price", lambda x: x.isna().sum())
    )
)

events["event_time"] = pd.to_datetime(
    events["event_time"],
    errors="coerce"
)

print("Некорректных дат:", events["event_time"].isna().sum())

In [ ]:
events_clean = events.merge(
    features[["listing_id", "published_at", "closed_at"]],
    on="listing_id",
    how="left",
    validate="many_to_one",
)

### 4.2. Уникальные покупатели и контактные действия
Контакты - начало чата и открытие номера телефона, а не подтверждённые звонки. Считаем количество разных пользователей и действий.

In [ ]:
contact_events = events_clean[
    events_clean["event_type"].isin(
        ["chat_start", "phone_view"]
    )
].copy()

buyer_features = (
    events_clean.groupby("listing_id")
    .agg(
        unique_buyers=("user_id", "nunique"),
        buyer_actions=("user_id", "count")
    )
)

contact_features = (
    contact_events.groupby("listing_id")
    .agg(
        unique_contact_buyers=("user_id", "nunique"),
        contact_actions=("user_id", "count")
    )
)

display(buyer_features.head())
display(contact_features.head())

### 4.3. Агрегация событий и временная структура
Считаем среднее число действий на пользователя, время до первого и после последнего контакта, длительность контактной активности. Счётчики для отсутствующих событий заполняются нулями, неопределённые интервалы остаются NaN.

In [ ]:
event_features = buyer_features.join(
    contact_features,
    how="left"
).fillna(0)

event_features["actions_per_buyer"] = safe_divide(
    event_features["buyer_actions"],
    event_features["unique_buyers"]
)

event_features["contacts_per_buyer"] = safe_divide(
    event_features["contact_actions"],
    event_features["unique_contact_buyers"]
)

contact_time = (
    contact_events.groupby("listing_id")
    .agg(
        first_contact=("event_time", "min"),
        last_contact=("event_time", "max"),
        published_at=("published_at", "first"),
        closed_at=("closed_at", "first")
    )
)

contact_time["hours_before_first_contact"] = (
    contact_time["first_contact"]
    - contact_time["published_at"]
).dt.total_seconds() / 3600

contact_time["hours_after_last_contact"] = (
    contact_time["closed_at"]
    - contact_time["last_contact"]
).dt.total_seconds() / 3600

contact_time["contact_activity_span"] = (
    contact_time["last_contact"]
    - contact_time["first_contact"]
).dt.total_seconds() / 3600

event_features = event_features.join(
    contact_time[
        [
            "hours_before_first_contact",
            "hours_after_last_contact",
            "contact_activity_span"
        ]
    ],
    how="left"
)

event_columns = event_features.columns.tolist()

features = features.drop(
    columns=event_columns,
    errors="ignore"
).merge(
    event_features.reset_index(),
    on="listing_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "unique_buyers",
    "buyer_actions",
    "unique_contact_buyers",
    "contact_actions"
]

features[count_columns] = (
    features[count_columns].fillna(0).astype(int)
)

print(features.shape)
print(features["listing_id"].is_unique)

### 4.4. Экспериментальные бинарные признаки
Тип последнего контакта и наличие обоих типов контактов. Оба признака остаются в полной таблице, но пока не включены в выбранный набор.

In [ ]:
# последнее действие - открытие телефона
events_dd_by_listing_id = contact_events.sort_values("event_time", ascending=False).drop_duplicates(subset="listing_id", keep="first")

phone_listing_ids = events_dd_by_listing_id.loc[
    events_dd_by_listing_id["event_type"] == "phone_view",
    "listing_id"
]

features["phone_view_is_last_event"] = (
    features["listing_id"].isin(phone_listing_ids)
).astype(int)

display(features["phone_view_is_last_event"].value_counts())

In [ ]:
contact_types = (
    contact_events.groupby("listing_id")["event_type"]
    .nunique()
)
both_contacts = contact_types[contact_types == 2]
features["has_both_contact_types"] = (
    features["listing_id"].isin(both_contacts.index)
).astype(int)

### Повторные контакты покупателей

Считаем повторные контакты и оба канала связи по паре объявления и покупателя. Для последних суток оставляем пропуск, если дата закрытия неизвестна.

In [26]:
contact_before_publication = contact_events["event_time"] < contact_events["published_at"]
contact_after_close = contact_events["event_time"] > contact_events["closed_at"]
buyer_contacts = contact_events.loc[~(contact_before_publication | contact_after_close)].copy()
contacts_by_buyer = buyer_contacts.groupby(["listing_id", "user_id"]).agg(
    contact_count=("event_type", "size"),
    contact_type_count=("event_type", "nunique"),
)
contacts_by_buyer["used_both_channels"] = contacts_by_buyer["contact_type_count"].eq(2)
contacts_by_buyer["contacted_again"] = contacts_by_buyer["contact_count"].gt(1)

buyer_contact_features = contacts_by_buyer.groupby("listing_id").agg(
    buyers_with_chat_and_phone=("used_both_channels", "sum"),
    repeat_contact_buyer_share=("contacted_again", "mean"),
)

# Открытие номера показывает интерес покупателя. О самом звонке данных нет.
hours_until_close = (
    buyer_contacts["closed_at"] - buyer_contacts["event_time"]
).dt.total_seconds() / 3600
recent_phone_views = buyer_contacts.loc[
    buyer_contacts["event_type"].eq("phone_view") & hours_until_close.between(0, 24)
]
phone_viewers_last_24h = recent_phone_views.groupby("listing_id")["user_id"].nunique()

listing_ids = features["listing_id"]
known_close = features.set_index("listing_id")["closed_at"].notna()
buyer_contact_features = buyer_contact_features.reindex(listing_ids)
buyer_contact_features["buyers_with_chat_and_phone"] = (
    buyer_contact_features["buyers_with_chat_and_phone"].fillna(0).astype(int)
)
buyer_contact_features["phone_viewers_last_24h"] = (
    phone_viewers_last_24h.reindex(listing_ids, fill_value=0).where(known_close)
)

print("Объявлений в таблице контактов:", len(buyer_contact_features))
display(buyer_contact_features.describe().round(2))

Объявлений в таблице контактов: 15274


       buyers_with_chat_and_phone  ...  phone_viewers_last_24h
count                    15274.00  ...                15234.00
mean                         0.03  ...                    0.14
std                          0.16  ...                    0.38
min                          0.00  ...                    0.00
25%                          0.00  ...                    0.00
50%                          0.00  ...                    0.00
75%                          0.00  ...                    0.00
max                          1.00  ...                    4.00

[8 rows x 3 columns]

## 5. Messages - структура переписок
### 5.1. Первичный анализ
Текст сообщений не используется в этом ноутбуке; исследуются количество, авторы и время отправки. Семантические признаки разрабатываются отдельно.

In [ ]:
display(messages.head(10))

print("Размер:", messages.shape)

print("\nТипы данных:")
print(messages.dtypes)

print("\nПропуски:")
print(messages.isna().sum())

print("\nДубликаты:")
print(messages.duplicated().sum())

print("\nАвторы сообщений:")
print(messages["author"].value_counts(dropna=False))

print("\nКоличество диалогов:")
print(messages["chat_id"].nunique())

print("\nКоличество объявлений:")
print(messages["listing_id"].nunique())

In [ ]:
# Распределение количества сообщений на один диалог.

chat_sizes = (
    messages.groupby(["listing_id", "chat_id"])
    .size()
)

display(chat_sizes.describe())

display(chat_sizes.value_counts().sort_index().head(30))

### 5.2. Время сообщений

Сообщения уже прошли проверку времени в `data_clearing.ipynb`. Добавляем дату закрытия для признаков активности перед снятием объявления. Пустой текст учитываем в статистике переписки.

In [ ]:
messages["sent_at"] = pd.to_datetime(messages["sent_at"], errors="coerce")

messages_clean = messages.merge(
    features[["listing_id", "closed_at"]],
    on="listing_id",
    how="left",
    validate="many_to_one",
)

### 5.3. Статистика на уровне чата
Группировка по listing_id и chat_id. Длинный диалог определён как восемь и более сообщений - исследовательская гипотеза на основе распределения длин.

In [ ]:
chat_stats = (
    messages_clean.groupby(["listing_id", "chat_id"])
    .agg(
        messages_count=("author", "size"),
        seller_messages=("author", lambda x: (x == "seller").sum()),
        buyer_messages=("author", lambda x: (x == "buyer").sum()),
        first_message=("sent_at", "min"),
        last_message=("sent_at", "max")
    )
)

display(chat_stats.head(10))

In [ ]:
chat_stats["is_unanswered"] = (
    chat_stats["seller_messages"] == 0
).astype(int)

chat_stats["is_long_chat"] = (
    chat_stats["messages_count"] >= 8
).astype(int)

chat_stats["chat_duration_hours"] = (
    chat_stats["last_message"]
    - chat_stats["first_message"]
).dt.total_seconds() / 3600

message_features = (
    chat_stats.groupby("listing_id")
    .agg(
        max_messages_in_chat=("messages_count", "max"),
        median_chat_length=("messages_count", "median"),
        std_chat_length=("messages_count", "std"),
        total_chat_messages=("messages_count", "sum"),
        seller_messages_total=("seller_messages", "sum"),
        unanswered_chats_share=("is_unanswered", "mean"),
        long_chats_count=("is_long_chat", "sum"),
        long_chats_share=("is_long_chat", "mean"),
        max_chat_duration=("chat_duration_hours", "max")
    )
)

message_features["dominant_chat_share"] = safe_divide(
    message_features["max_messages_in_chat"],
    message_features["total_chat_messages"]
)

message_features["seller_message_share"] = safe_divide(
    message_features["seller_messages_total"],
    message_features["total_chat_messages"]
)

### 5.4. Присоединение признаков диалогов
Агрегаты на уровне объявления присоединяются по listing_id; нулями заполняются только счётчики.

In [ ]:
message_columns = message_features.columns.tolist()

features = features.drop(
    columns=message_columns,
    errors="ignore"
).merge(
    message_features.reset_index(),
    on="listing_id",
    how="left",
    validate="one_to_one"
)

message_count_columns = [
    "max_messages_in_chat",
    "total_chat_messages",
    "seller_messages_total",
    "long_chats_count"
]

features[message_count_columns] = (
    features[message_count_columns].fillna(0).astype(int)
)


print(features.shape)
print(features["listing_id"].is_unique)

### 5.5. Скорость ответа продавца
Сортируем сообщения внутри диалога; `shift(1)` даёт предыдущего автора и время. Время ответа считается от непосредственно предшествующего сообщения покупателя, а не от первого сообщения серии.

In [ ]:
messages_sorted = messages_clean.sort_values(
    ["listing_id", "chat_id", "sent_at"]
).copy()

message_groups = messages_sorted.groupby(
    ["listing_id", "chat_id"]
)

messages_sorted["prev_author"] = (
    message_groups["author"].shift(1)
)

messages_sorted["prev_time"] = (
    message_groups["sent_at"].shift(1)
)

seller_replies = messages_sorted[
    (messages_sorted["author"] == "seller")
    & (messages_sorted["prev_author"] == "buyer")
].copy()

seller_replies["reply_hours"] = (
    seller_replies["sent_at"]
    - seller_replies["prev_time"]
).dt.total_seconds() / 3600

features["median_seller_reply_hours"] = (
    features["listing_id"].map(
        seller_replies.groupby("listing_id")["reply_hours"]
        .median()
    )
)

### 5.6. Активность переписки перед закрытием

Время после последнего сообщения и число сообщений за последние 24 часа. Если дата закрытия неизвестна, оба признака остаются пропусками.

In [ ]:
messages_clean["hours_until_close"] = (
    messages_clean["closed_at"]
    - messages_clean["sent_at"]
).dt.total_seconds() / 3600

features["hours_after_last_message"] = (
    features["listing_id"].map(
        messages_clean.groupby("listing_id")
        ["hours_until_close"].min()
    )
)

recent_messages = messages_clean[
    messages_clean["hours_until_close"] <= 24
]

features["messages_last_24h"] = (
    features["listing_id"]
    .map(recent_messages.groupby("listing_id").size())
    .fillna(0)
    .astype(int)
    .where(features["closed_at"].notna())
)

print(features.shape)
print(features["listing_id"].is_unique)

### Этапы разговора

Учитываем, кто написал реплику и что произошло дальше в том же чате. Сообщение о приезде, после которого покупатель отказался, теряет силу. Фраза продавца «уже продано» остаётся отдельным сигналом, поскольку покупатель по ней неизвестен.

In [35]:
chat_keys = ["listing_id", "chat_id"]
dialogue_messages = messages_clean.join(
    features.set_index("listing_id")["published_at"],
    on="listing_id",
    validate="many_to_one",
)
before_publication = dialogue_messages["sent_at"] < dialogue_messages["published_at"]
after_close = dialogue_messages["sent_at"] > dialogue_messages["closed_at"]
dialogue_messages = dialogue_messages.loc[~(before_publication | after_close)].copy()

normalized_text = (
    dialogue_messages["text"]
    .str.casefold()
    .str.replace("ё", "е")
    .str.replace(r"[^а-яa-z]+", " ", regex=True)
    .str.strip()
)
buyer = dialogue_messages["author"].eq("buyer")
seller = dialogue_messages["author"].eq("seller")
dialogue_messages["buyer_arrival"] = buyer & normalized_text.str.contains(
    r"^(?:еду\b|выезжаю\b|подъехал\b|я на месте\b|уже подхожу\b)", na=False
)
dialogue_messages["seller_meeting"] = seller & normalized_text.str.contains(
    r"^(?:спускаюсь\b|выхожу\b|сейчас выйду\b)", na=False
)
dialogue_messages["buyer_thanks"] = buyer & normalized_text.isin([
    "все супер спасибо", "спасибо все отлично",
])
dialogue_messages["buyer_refusal"] = buyer & normalized_text.str.contains(
    r"купил в другом месте|не приеду|не смогу приехать|извините передумал|"
    r"не возьму|посмотрел не подошло|не то что ожидал|^дорого спасибо$",
    na=False,
)
dialogue_messages["seller_sold"] = seller & normalized_text.isin([
    "продано", "уже продано", "извините уже продал", "уже нет извините",
])

message_signals = [
    "buyer_arrival", "seller_meeting", "buyer_thanks", "buyer_refusal", "seller_sold",
]
display(dialogue_messages[message_signals].sum().rename("Реплик"))

buyer_arrival      6668
seller_meeting     2563
buyer_thanks        688
buyer_refusal     10914
seller_sold        3168
Name: Реплик, dtype: int64

In [36]:
dialogue_flags = dialogue_messages.groupby(chat_keys)[message_signals].any()
arrival_times = (
    dialogue_messages.loc[dialogue_messages["buyer_arrival"]]
    .groupby(chat_keys)["sent_at"].max().reindex(dialogue_flags.index)
)
refusal_times = (
    dialogue_messages.loc[dialogue_messages["buyer_refusal"]]
    .groupby(chat_keys)["sent_at"].max().reindex(dialogue_flags.index)
)
thanks_times = (
    dialogue_messages.loc[dialogue_messages["buyer_thanks"]]
    .groupby(chat_keys)["sent_at"].max().reindex(dialogue_flags.index)
)

dialogue_flags["arrival_without_later_refusal"] = (
    dialogue_flags["buyer_arrival"] & ~refusal_times.gt(arrival_times)
)
dialogue_flags["mutual_meeting"] = (
    dialogue_flags["arrival_without_later_refusal"] & dialogue_flags["seller_meeting"]
)
dialogue_flags["handoff_thanks"] = (
    dialogue_flags["arrival_without_later_refusal"] & thanks_times.gt(arrival_times)
)

dialogue_features = dialogue_flags.groupby("listing_id").agg(
    has_arrival_without_later_refusal=("arrival_without_later_refusal", "any"),
    has_mutual_meeting=("mutual_meeting", "any"),
    has_handoff_thanks=("handoff_thanks", "any"),
    buyer_refusal_share=("buyer_refusal", "mean"),
    has_seller_sold=("seller_sold", "any"),
)

# У объявления без чатов флаги равны нулю, а доля отказов неизвестна.
dialogue_flag_columns = dialogue_features.select_dtypes(include="bool").columns
dialogue_flags_by_listing = dialogue_features[dialogue_flag_columns].reindex(
    features["listing_id"], fill_value=False
).astype(int)
dialogue_features = dialogue_features.reindex(features["listing_id"])
dialogue_features[dialogue_flag_columns] = dialogue_flags_by_listing

print("Объявлений в таблице диалогов:", len(dialogue_features))
display(dialogue_features.describe().round(2))

Объявлений в таблице диалогов: 15274


       has_arrival_without_later_refusal  ...  has_seller_sold
count                            15274.0  ...         15274.00
mean                                 0.2  ...             0.11
std                                  0.4  ...             0.32
min                                  0.0  ...             0.00
25%                                  0.0  ...             0.00
50%                                  0.0  ...             0.00
75%                                  0.0  ...             0.00
max                                  1.0  ...             1.00

[8 rows x 5 columns]

### Анализ истории продавцов

In [ ]:
print("Всего объявлений:", len(features))
print("Уникальных продавцов:", features["seller_id"].nunique())

print("\nТипы закрытия объявлений:")
display(features["close_type"].value_counts())

seller_listings = (
    features.groupby("seller_id")["listing_id"]
    .nunique()
)

print("\nСтатистика объявлений на продавца:")
display(seller_listings.describe())

print("\nКоличество продавцов по числу объявлений:")
display(seller_listings.value_counts().sort_index())

print("\nПродавцов с несколькими объявлениями:")
print((seller_listings > 1).sum())

In [ ]:
display(
    research.groupby("close_type")
    .agg(
        total=("is_deal", "count"),
        deals=("is_deal", "sum"),
        deal_rate=("is_deal", "mean")
    )
)

### Формирование исторических признаков продавцов

In [ ]:
seller_history = features[
    ["listing_id", "seller_id", "published_at"]
].merge(
    features[
        ["listing_id", "seller_id", "closed_at", "close_type"]
    ],
    on="seller_id",
    suffixes=("", "_previous")
)

# Учитываем только ранее закрытые объявления продавца
seller_history = seller_history[
    (seller_history["listing_id"] !=
     seller_history["listing_id_previous"])
    & (seller_history["closed_at"] <
       seller_history["published_at"])
].copy()

display(seller_history.head())

In [ ]:
seller_history["is_blocked"] = (
    seller_history["close_type"] == "заблокировано"
).astype(int)

seller_history["is_expired"] = (
    seller_history["close_type"] == "истёк срок"
).astype(int)

seller_history["is_removed"] = (
    seller_history["close_type"] == "снято продавцом"
).astype(int)

seller_stats = seller_history.groupby("listing_id").agg(
    seller_previous_listings=("listing_id_previous", "count"),
    seller_previous_blocks=("is_blocked", "sum"),
    seller_previous_expired=("is_expired", "sum"),
    seller_previous_removed=("is_removed", "sum")
)

In [ ]:
for col in seller_stats.columns:
    features[col] = (
        features["listing_id"]
        .map(seller_stats[col])
        .fillna(0)
        .astype(int)
    )

In [ ]:
features["seller_block_rate"] = safe_divide(
    features["seller_previous_blocks"],
    features["seller_previous_listings"]
)

features["seller_has_block_history"] = (
    features["seller_previous_blocks"] > 0
).astype(int)

features["seller_expired_rate"] = safe_divide(
    features["seller_previous_expired"],
    features["seller_previous_listings"]
)

## 6. Подготовка таблицы для обучения
### 6.1. Дополнительное кодирование категорий для экспорта
Ручные индикаторы сохраняем в полной таблице для команды. В контролируемых экспериментах используются исходные четыре категориальных столбца, а OneHotEncoder обучается внутри каждого фолда. Индикаторы не добавляются в отобранный набор.

In [ ]:
# seller_type
seller_type_counts = features["seller_type"].value_counts()
for seller_type in seller_type_counts.index.tolist():
    features[f"seller_type_{seller_type}"] = (
        features["seller_type"] == seller_type
    ).astype(int)

# category
category_type_counts = features["category"].value_counts()
for category_type in category_type_counts.index.tolist():
    features[f"category_type_{category_type}"] = (
        features["category"] == category_type
    ).astype(int)

# region
region_type_counts = features["region"].value_counts()
for region_type in region_type_counts.index.tolist():
    features[f"region_type_{region_type}"] = (
        features["region"] == region_type
    ).astype(int)

# close_type
close_type_counts = features["close_type"].value_counts()
for close_type in close_type_counts.index.tolist():
    features[f"close_type_{close_type}"] = (
        features["close_type"] == close_type
    ).astype(int)

### 6.2. Размеченная выборка и проверка сигналов
Единое финальное объединение с verified. Предыдущие временные объединения в EDA предназначены только для анализа цен и времени жизни; именно этот research используется во всех экспериментах.

In [ ]:
research = features.merge(
    verified,
    on="listing_id",
    how="inner",
    validate="one_to_one"
)

eda_behavior_features = [
    "messages_per_chat",
    "chats_per_view",
    "phones_per_view",
    "favorites_per_view",
    "contacts_per_view",
    "views_per_day",
    "chats_per_day",
    "messages_per_day",
    "contacts_per_day",
    "no_contacts",
    "no_chats",
    "no_phone_views"
]

display(
    research.groupby("is_deal")[eda_behavior_features]
    .median()
    .T
)

correlations = (
    research.select_dtypes(include="number")
    .drop(columns=["listing_id", "seller_id"])
    .corr()["is_deal"]
    .drop("is_deal")
    .sort_values(key=abs, ascending=False)
)

display(correlations)

In [ ]:
assert len(features) == len(listings), "Потеряны или размножены объявления"
assert features["listing_id"].is_unique
assert features.columns.is_unique
assert len(research) == len(verified_clean)
assert "is_deal" not in features.columns
print("Объявлений:", len(features))
print("Размеченных:", len(research))
display(research["is_deal"].value_counts())

## 7. Модель и эксперименты
### 7.1. Единый протокол оценки
Логистическая регрессия, 5-fold StratifiedKFold с seed=42, стандартный порог 0.5. Медианная импутация и масштабирование чисел, импутация и OneHotEncoder категорий обучаются внутри фолда. Сравниваем средний F1, разброс F1, precision, recall и ROC-AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


def evaluate_features(numeric, categorical):
    """Оценка выбранных признаков на research по фиксированным фолдам."""

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    X = research[numeric + categorical]
    y = research["is_deal"]

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["f1", "precision", "recall", "roc_auc"],
        error_score="raise"
    )

    return {
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    }

### 7.2. Listings: базовый набор и группы признаков

In [ ]:
base_numeric = [
    "price",
    "current_price",
    "photos_count",
    "description_length",
    "views_total",
    "favorites_total",
    "phone_views_total",
    "chats_total",
    "messages_total"
]

categorical = [
    "seller_type",
    "category",
    "region",
    "close_type"
]

price_features = [
    "log_relative_price",
    "relative_current_price",
    "discount_pct",
    "has_discount",
    "log_price"
]

time_features = [
    "lifetime_days",
    "log_lifetime",
    "is_closed_first_week"
]

In [ ]:
behavior_features = [
    "messages_per_chat",
    "chats_per_view",
    "phones_per_view",
    "favorites_per_view",
    "contacts_per_view",
    "views_per_day",
    "chats_per_day",
    "messages_per_day",
    "contacts_per_day",
    "no_contacts",
    "no_chats",
    "no_phone_views",
    "relative_messages_per_chat",
    "relative_description_length"
]

binary_features = [
    "is_used",
    "is_new",
    "is_like_new",
    "has_documents",
    "is_negotiable",
    "is_urgent",
    "is_excellent",
    "is_good_condition",
    "description_missing",
    "is_overpriced",
    "is_closed_first_day",
    "no_views"
]

# Выбранный набор Listings без экспериментальных бинарных признаков.
all_features = (
    base_numeric
    + price_features
    + time_features
    + behavior_features
)

# Основные эксперименты
experiments = {
    "Baseline": base_numeric,
    "Baseline + Price": base_numeric + price_features,
    "Baseline + Time": base_numeric + time_features,
    "Baseline + Behavior": base_numeric + behavior_features,
    "Baseline + Binary": base_numeric + binary_features,
    "All features without binary": all_features,
    "All features with binary": all_features + binary_features
}

results = pd.DataFrame({
    name: evaluate_features(cols, categorical)
    for name, cols in experiments.items()
}).T

display(results.round(4))

### 7.3. Listings: индивидуальные бинарные признаки
Проверяем вклад каждого признака отдельно. Не выбираем их только по небольшим колебаниям F1.

In [ ]:
# Проверка каждого бинарного признака отдельно
binary_results = {}

binary_results["Without binary"] = evaluate_features(
    all_features, categorical
)

for feature in binary_features:
    binary_results[feature] = evaluate_features(
        all_features + [feature],
        categorical
    )

binary_results = pd.DataFrame(binary_results).T

# Считаем изменение F1 относительно лучшей модели
binary_results["F1_delta"] = (
    binary_results["F1"]
    - binary_results.loc["Without binary", "F1"]
)

display(binary_results.sort_values("F1_delta", ascending=False).round(4))

### 7.4. Surveys: оценка ответов и дополнительных флагов

In [ ]:
survey_experiments = {
    "Original": all_features,

    "With survey answers": (
        all_features + survey_features
    ),

    "With all survey features": (
        all_features
        + survey_features
        + ["survey_eligible", "survey_answered"]
    )
}

survey_results = pd.DataFrame({
    name: evaluate_features(cols, categorical)
    for name, cols in survey_experiments.items()
}).T

display(survey_results.round(4))

In [ ]:
# Набор после добавления опросов.

best_features = (
    all_features
    + survey_features
    + ["survey_eligible", "survey_answered"]
)

### 7.5. Events: оценка групп признаков и индивидуальных вкладов

In [ ]:
event_buyer_features = [
    "unique_buyers",
    "unique_contact_buyers",
    "actions_per_buyer",
    "contacts_per_buyer"
]

event_time_features = [
    "hours_before_first_contact",
    "hours_after_last_contact",
    "contact_activity_span"
]

event_experiments = {
    "Original": best_features,

    "With buyers": (
        best_features + event_buyer_features
    ),

    "With event time": (
        best_features + event_time_features
    ),

    "With all events": (
        best_features
        + event_buyer_features
        + event_time_features
    )
}

event_results = pd.DataFrame({
    name: evaluate_features(cols, categorical)
    for name, cols in event_experiments.items()
}).T

display(event_results.round(4))

In [ ]:
event_individual_results = pd.DataFrame({
    col: evaluate_features(
        best_features + [col],
        categorical
    )
    for col in event_buyer_features + event_time_features
}).T

event_individual_results["F1_gain"] = (
    event_individual_results["F1"]
    - event_results.loc["Original", "F1"]
)

display(
    event_individual_results
    .sort_values("F1_gain", ascending=False)
    .round(4)
)

In [ ]:
# Набор после добавления событий покупателей.

best_with_events = (
    best_features
    + event_buyer_features
    + event_time_features
)

### 7.6. Events: проверка наличия обоих типов контактов
Сравниваем с набором Listings и с набором после Events; проверяем дублирование информации из no_chats и no_phone_views.

In [ ]:
display(pd.DataFrame({
    "Original": evaluate_features(
        all_features, categorical
    ),
    "With both contact types": evaluate_features(
        all_features + ["has_both_contact_types"],
        categorical
    )
}).T.round(4))

In [ ]:
display(pd.DataFrame({
    "Original": evaluate_features(
        best_with_events, categorical
    ),
    "With both contact types": evaluate_features(
        best_with_events + ["has_both_contact_types"],
        categorical
    )
}).T.round(4))

In [ ]:
display(research["has_both_contact_types"].value_counts())

display(pd.crosstab(
    research["has_both_contact_types"],
    [research["no_chats"], research["no_phone_views"]]
))

### 7.7. Messages: оценка структуры диалогов и активности авторов

In [ ]:
message_structure_features = [
    "max_messages_in_chat",
    "median_chat_length",
    "std_chat_length",
    "long_chats_count",
    "long_chats_share",
    "max_chat_duration",
    "dominant_chat_share"
]

message_author_features = [
    "unanswered_chats_share",
    "seller_message_share"
]

message_experiments = {
    "Original": best_with_events,

    "Chat structure": (
        best_with_events + message_structure_features
    ),

    "Author activity": (
        best_with_events + message_author_features
    ),

    "All messages": (
        best_with_events
        + message_structure_features
        + message_author_features
    )
}

message_results = pd.DataFrame({
    name: evaluate_features(cols, categorical)
    for name, cols in message_experiments.items()
}).T

display(message_results.round(4))

In [ ]:
# Набор после добавления структуры сообщений.

best_with_messages = (
    best_with_events
    + message_structure_features
    + message_author_features
)

### 7.8. Messages: оценка временных признаков
Эксперимент выполняется только после объявления best_with_messages.

In [ ]:
message_time_features = [
    "median_seller_reply_hours",
    "hours_after_last_message",
    "messages_last_24h"
]

message_time_results = pd.DataFrame({
    "Original": evaluate_features(
        best_with_messages, categorical
    ),
    "With message time": evaluate_features(
        best_with_messages + message_time_features,
        categorical
    )
}).T

display(message_time_results.round(4))

## 8. Итоговый набор и экспорт
### 8.1. Выбор признаков
По сохранённым результатам исходного ноутбука (до реорганизации) добавление временных признаков повысило F1 с 0.7682 до 0.7730 и снизило стандартное отклонение F1, при этом ROC-AUC снизился с 0.9079 до 0.9029. Поэтому предварительно включаем все три временных признака. Это не независимая оценка.

| Этап | F1 | F1_std | ROC-AUC |
|---|---:|---:|---:|
| Baseline | 0.6040 | 0.0544 | 0.7730 |
| Listings | 0.7256 | 0.0350 | 0.8425 |
| + Surveys | 0.7284 | 0.0344 | 0.8715 |
| + Events | 0.7519 | 0.0274 | 0.8927 |
| + Messages (структура и авторы) | 0.7682 | 0.0453 | 0.9079 |
| + Messages (время) | 0.7730 | 0.0302 | 0.9029 |

После Run All ориентируемся на вновь рассчитанные результаты, а не на эти исторические значения. CatShareMAE, настройка порога и обработка аномалий остаются отдельными задачами.

In [ ]:
best_final_features = best_with_messages + message_time_features
final_feature_columns = best_final_features + categorical
assert len(final_feature_columns) == len(set(final_feature_columns))
assert set(final_feature_columns).issubset(features.columns)
assert not {"listing_id", "seller_id", "is_deal"}.intersection(final_feature_columns)

# Сводка результатов текущего запуска (без дополнительного переобучения).
progress_results = pd.DataFrame({
    "Baseline": results.loc["Baseline"],
    "Listings": results.loc["All features without binary"],
    "+ Surveys": survey_results.loc["With all survey features"],
    "+ Events": event_results.loc["With all events"],
    "+ Messages": message_results.loc["All messages"],
    "+ Message time": message_time_results.loc["With message time"]
}).T
display(progress_results.round(4))
print("Отобранных признаков:", len(final_feature_columns))

## Оптимизация модели
### Исследование порога классификации

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score

final_numeric = (
    best_with_messages + message_time_features
)

X = research[final_numeric + categorical]
y = research["is_deal"]

thresholds = np.arange(0.20, 0.71, 0.01)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score

final_numeric = best_with_messages + message_time_features

X = research[final_numeric + categorical]
y = research["is_deal"]

def build_model(C=1.0):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, final_numeric),
        ("cat", categorical_pipeline, categorical)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, C=C))
    ])

In [ ]:
probabilities = cross_val_predict(
    build_model(),
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

In [ ]:
threshold_results = []

for threshold in np.arange(0.20, 0.71, 0.01):
    predictions = (probabilities >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "F1": f1_score(y, predictions),
        "Precision": precision_score(y, predictions),
        "Recall": recall_score(y, predictions)
    })

threshold_results = pd.DataFrame(threshold_results)

display(
    threshold_results
    .sort_values("F1", ascending=False)
    .head(10)
    .round(4)
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(threshold_results["threshold"],
         threshold_results["F1"], label="F1")

plt.plot(threshold_results["threshold"],
         threshold_results["Precision"], label="Precision")

plt.plot(threshold_results["threshold"],
         threshold_results["Recall"], label="Recall")

plt.axvline(0.5, linestyle="--", color="gray")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold optimization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 8.2. Валидация оптимизированного порога

In [ ]:
from sklearn.model_selection import StratifiedKFold

inner_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

nested_results = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Подбираем порог только на обучающих данных
    train_probs = cross_val_predict(
        build_model(),
        X_train,
        y_train,
        cv=inner_cv,
        method="predict_proba"
    )[:, 1]

    best_threshold = max(
        np.arange(0.20, 0.71, 0.01),
        key=lambda t: f1_score(
            y_train, (train_probs >= t).astype(int)
        )
    )

    # Обучаем модель и проверяем на внешнем фолде
    model = build_model()
    model.fit(X_train, y_train)

    test_probs = model.predict_proba(X_test)[:, 1]

    nested_results.append({
        "fold": fold,
        "threshold": best_threshold,
        "F1_original": f1_score(
            y_test, (test_probs >= 0.5).astype(int)
        ),
        "F1_optimized": f1_score(
            y_test, (test_probs >= best_threshold).astype(int)
        )
    })

nested_results = pd.DataFrame(nested_results)

display(nested_results.round(4))
display(nested_results.mean().round(4))

### 8.3. Исследование силы регуляризации

In [ ]:
C_values = [
    0.001, 0.003,
    0.01, 0.03,
    0.1, 0.3,
    1.0, 3.0,
    10.0, 30.0, 100.0
]

regularization_results = []

for C in C_values:

    scores = cross_validate(
        build_model(C=C),
        X,
        y,
        cv=cv,
        scoring=["f1", "precision", "recall", "roc_auc"]
    )

    regularization_results.append({
        "C": C,
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

regularization_results = pd.DataFrame(
    regularization_results
)

display(
    regularization_results
    .sort_values("F1", ascending=False)
    .round(4)
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    regularization_results["C"],
    regularization_results["F1"],
    marker="o",
    label="F1"
)

plt.axhline(
    0.7730,
    color="gray",
    linestyle="--",
    label="Current baseline"
)

plt.xscale("log")
plt.xlabel("C")
plt.ylabel("F1")
plt.title("Regularization strength optimization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 8.4. Сравнение Logistic Regression и CatBoost

In [ ]:
from catboost import CatBoostClassifier

# Подготовка данных
X_cat = X.copy()

X_cat[final_numeric] = X_cat[final_numeric].replace(
    [np.inf, -np.inf], np.nan
)

for col in categorical:
    X_cat[col] = (
        X_cat[col].astype("string")
        .fillna("unknown")
        .astype(str)
    )

# Создание модели
catboost_model = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.03,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Кросс-валидация
catboost_scores = cross_validate(
    catboost_model,
    X_cat,
    y,
    cv=cv,
    scoring=["f1", "precision", "recall", "roc_auc"],
    params={"cat_features": categorical},
    error_score="raise"
)

# Расчёт метрик
catboost_results = {
    "F1": catboost_scores["test_f1"].mean(),
    "F1_std": catboost_scores["test_f1"].std(),
    "Precision": catboost_scores["test_precision"].mean(),
    "Recall": catboost_scores["test_recall"].mean(),
    "ROC_AUC": catboost_scores["test_roc_auc"].mean()
}

# Сравнение моделей
display(pd.DataFrame({
    "Logistic Regression": evaluate_features(
        final_numeric, categorical
    ),
    "CatBoost": catboost_results
}).T.round(4))

### 8.5. Проверка исторических признаков продавцов

In [ ]:
seller_block_features = [
    "seller_previous_blocks",
    "seller_block_rate",
    "seller_has_block_history"
]

seller_activity_features = [
    "seller_previous_listings",
    "seller_previous_expired",
    "seller_previous_removed",
    "seller_expired_rate"
]

seller_experiments = {
    "Original": final_numeric,

    "Block history": (
        final_numeric + seller_block_features
    ),

    "Seller activity": (
        final_numeric + seller_activity_features
    ),

    "All seller features": (
        final_numeric
        + seller_block_features
        + seller_activity_features
    )
}

seller_results = pd.DataFrame({
    name: evaluate_features(cols, categorical)
    for name, cols in seller_experiments.items()
}).T

display(seller_results.round(4))

In [ ]:
print("Размеченных объявлений с историей продавца:")
print((research["seller_previous_listings"] > 0).sum())

print("\nРазмеченных объявлений с историей блокировок:")
print(research["seller_has_block_history"].sum())

### 8.2. Сохранение CSV
Перед экспортом присоединяем признаки из `chat_features.csv` по `listing_id`. Суммируем число оценённых сообщений, берём максимальный сигнал каждого знака и пересчитываем итоговый балл и силу сигнала для объявления. Для объявлений без оценённых чатов используем нули. Сохраняем эти признаки в обеих таблицах.

Модель использует features.csv. В этот экспорт добавляем этапы разговора и повторные контакты покупателей. При экспорте убираем число длинных чатов и долю сообщений в самом длинном чате. Вместе их исключение повысило F1 и снизило ошибку долей по категориям. Исследовательские расчёты выше используют исходные признаки. В сокращённую таблицу входят ID, отобранные ранее признаки и признаки из текста чатов. Разметка не сохраняется в таблицах признаков. Сохраняем также обработанные surveys, events и messages с исходными столбцами последних двух таблиц.

In [ ]:
chat_features = pd.read_csv("chat_features.csv")
listing_chat_features = chat_features.groupby("listing_id").agg(
    scored_message_count=("scored_message_count", "sum"),
    strongest_positive_signal=("strongest_positive_signal", "max"),
    strongest_negative_signal=("strongest_negative_signal", "max"),
)
listing_chat_features["deal_signal_score"] = (
    listing_chat_features["strongest_positive_signal"]
    - listing_chat_features["strongest_negative_signal"]
)
listing_chat_features["deal_signal_strength"] = listing_chat_features[
    ["strongest_positive_signal", "strongest_negative_signal"]
].max(axis=1)

chat_feature_columns = listing_chat_features.columns.tolist()
features = features.drop(columns=chat_feature_columns, errors="ignore").join(
    listing_chat_features, on="listing_id", how="left", validate="one_to_one"
)
features[chat_feature_columns] = features[chat_feature_columns].fillna(0)
features["scored_message_count"] = features["scored_message_count"].astype(int)

In [75]:
OUTPUT = Path("data_by_dewallen")
OUTPUT.mkdir(exist_ok=True)

best_export_features = final_feature_columns + chat_feature_columns
excluded_features = ["long_chats_count", "dominant_chat_share"]
model_features = features.drop(columns=excluded_features).merge(
    dialogue_features.reset_index(), on="listing_id", how="left", validate="one_to_one"
).merge(
    buyer_contact_features.reset_index(), on="listing_id", how="left", validate="one_to_one"
)

assert model_features["listing_id"].is_unique, "Повторяются ID объявлений"
assert len(model_features) == len(listings), "Изменилось число объявлений"
assert not model_features.columns.isin(excluded_features).any(), "Лишние признаки в экспорте"

model_features.to_csv(OUTPUT / "features.csv", index=False)
print("Столбцов в features.csv:", model_features.shape[1])
features[["listing_id"] + best_export_features].to_csv(
    OUTPUT / "best_features.csv", index=False
)
surveys.to_csv(OUTPUT / "surveys.csv", index=False)
events_clean[events.columns].to_csv(OUTPUT / "events.csv", index=False)
messages_clean[messages.columns].to_csv(OUTPUT / "messages.csv", index=False)

for path in sorted(OUTPUT.glob("*.csv")):
    print("Сохранено:", path)
print("Папка:", OUTPUT.resolve())

Столбцов в features.csv: 133
Сохранено: data_by_dewallen/best_features.csv
Сохранено: data_by_dewallen/events.csv
Сохранено: data_by_dewallen/features.csv
Сохранено: data_by_dewallen/messages.csv
Сохранено: data_by_dewallen/surveys.csv
Папка: /Users/ynelo/Documents/github/neuroslop_kygarikovs_hakaton/data_by_dewallen


### 8.3. Контроль экспортированных таблиц
Повторно читаем CSV и проверяем размеры и ключи. Для сравнения качества используем командный модуль `f1.py` (или его исходное имя `f1_check.py`, если файл ещё не переименован). Полная таблица включает также заголовки, даты и дублирующие столбцы; такой эксперимент не является чистым сравнением инженерных признаков. Если модуль отсутствует, основной ноутбук всё равно завершает работу.

In [76]:
full_df = pd.read_csv(OUTPUT / "features.csv")
best_df = pd.read_csv(OUTPUT / "best_features.csv")
assert full_df["listing_id"].is_unique and best_df["listing_id"].is_unique
assert full_df["listing_id"].equals(best_df["listing_id"])
assert len(best_df) == len(listings)
assert best_df.columns.tolist() == ["listing_id"] + best_export_features

full_df = full_df.merge(verified, on="listing_id", how="inner", validate="one_to_one")
best_df = best_df.merge(verified, on="listing_id", how="inner", validate="one_to_one")
print("All features (столбцы без ID и target):", full_df.drop(columns=["listing_id", "seller_id", "is_deal"], errors="ignore").shape[1])
print("Best features:", len(best_export_features))

from importlib.util import find_spec

if find_spec("f1") is not None:
    from f1 import evaluate_features as check_f1
elif find_spec("f1_check") is not None:
    from f1_check import evaluate_features as check_f1
else:
    check_f1 = None
    print("Командный модуль f1.py не найден. Сравнение CSV пропущено.")

if check_f1 is not None:
    export_results = pd.DataFrame({
        "All features": check_f1(full_df),
        "Best features": check_f1(best_df)
    }).T
    display(export_results.round(4))

All features (столбцы без ID и target): 131
Best features: 65


                   F1  F1_std  Precision  Recall  ROC_AUC
All features   0.8163  0.0381     0.8555  0.7844    0.918
Best features  0.7559  0.0308     0.7952  0.7243    0.887

### 8.4. Проверка итоговой модели

Запустите `model.ipynb` для оценки на пяти фолдах, расчёта ошибки долей по категориям и сохранения `submission.csv`.